# 智能手套ASL识别系统 - 模型部署模块

## 项目概述

本项目是一个基于机器学习的智能手套系统，用于识别美国手语(ASL)并将其转换为文字/语音输出。

### 系统架构

```
┌─────────────┐      WiFi/TCP      ┌─────────────────┐      ┌─────────────┐
│   ESP32     │  ═══════════════►  │   树莓派/PC     │ ──►  │  文字/语音   │
│  (传感器)    │    44字节数据包     │  (本Notebook)   │      │   输出      │
└─────────────┘                    └─────────────────┘      └─────────────┘
```

### 核心组件

1. **硬件层**: ESP32 + 5个弯曲传感器 + MPU6050(陀螺仪+加速度计)
2. **通信层**: TCP Socket (端口4444)
3. **模型层**: 预训练的机器学习模型(Random Forest/其他分类器)
4. **输出层**: 文字显示 + 可选语音合成(TTS)

### 数据流说明

- ESP32以44字节数据包格式发送传感器数据
- 数据包结构: 5个弯曲值(20字节) + 3轴陀螺仪(12字节) + 3轴加速度计(12字节)
- 系统接收数据后进行预处理和标准化
- 机器学习模型进行实时预测
- 输出识别的手语字母/手势


## 1. 导入必要的库和依赖

本单元格导入模型部署所需的所有Python库。

### 库说明:

| 库名 | 用途 | 版本建议 |
|------|------|----------|
| `socket` | TCP网络通信，接收ESP32数据 | 内置 |
| `pandas` | 数据处理和DataFrame操作 | >=1.3.0 |
| `joblib` | 加载预训练的模型文件(.pkl) | >=1.0.0 |
| `pyttsx3` | 文字转语音(TTS)输出 | >=2.90 |
| `sklearn` | 数据标准化和模型推理 | >=1.0.0 |
| `struct` | 解析二进制数据包 | 内置 |
| `logging` | 日志记录和调试 | 内置 |

### 安装依赖命令(如需要):

```bash
pip install pandas joblib pyttsx3 scikit-learn
```

In [4]:
'''
Author: chouyougongchang233 megakidney@qq.com
Date: 2026-02-15 22:23:36
LastEditors: chouyougongchang233 megakidney@qq.com
LastEditTime: 2026-02-17 22:01:52
FilePath: \AlgeriaSmartGloves\Smart-glove-main\model_deployment.ipynb
Description: 这是默认设置,请设置`customMade`, 打开koroFileHeader查看配置 进行设置: https://github.com/OBKoro1/koro1FileHeader/wiki/%E9%85%8D%E7%BD%AE
'''
# ==========================================
# 第1部分: 导入必要的库和依赖
# ==========================================

import socket          # TCP/UDP网络通信库，用于与ESP32建立Socket连接
import pandas as pd    # 数据处理库，用于构建特征DataFrame
from joblib import load  # 模型加载工具，用于加载.pkl格式的模型文件
import pyttsx3       # 文字转语音库，用于语音输出识别结果
import time          # 时间控制库，用于设置采样间隔
from sklearn.preprocessing import StandardScaler  # 数据标准化工具(虽然这里主要是用已保存的scaler)
import struct        # 二进制数据解析库，用于解码ESP32发送的float数据
from sklearn.metrics import accuracy_score  # 模型评估指标(预留)
import logging       # 日志记录库，用于调试和错误追踪
import subprocess    # 系统命令执行库(预留)

print("[INFO] 所有依赖库导入成功")

[INFO] 所有依赖库导入成功


## 2. 模型和组件加载

本单元格加载预训练的机器学习模型及其相关组件。

### 加载的文件说明:

1. **模型文件 (`tarek.pkl`)**: 训练好的分类模型(如Random Forest、SVM等)
   - 用途: 根据输入特征预测手语类别
   - 输入: 11维特征向量 (5个弯曲值 + 3轴陀螺仪 + 3轴加速度计)
   - 输出: 编码后的类别标签

2. **标准化器 (`scaler81.pkl`)**: MinMaxScaler或StandardScaler对象
   - 用途: 对输入特征进行标准化/归一化处理
   - **重要**: 必须使用训练时相同的标准化参数

3. **标签编码器 (`labelencoder81.pkl`)**: LabelEncoder对象
   - 用途: 将数值预测结果转换为原始标签(A, B, C... 或 单词)

### 文件路径配置:

默认路径是基于树莓派的挂载点 `/media/raspberry/64E5-D9E51/`，
在Windows上需要修改为本地路径，如 `H:/models/`。

In [ ]:
# ==========================================
# 第2部分: 加载预训练模型和组件
# ==========================================

# TODO: 根据实际环境修改模型文件路径
# 当前路径是基于树莓派的USB挂载点，Windows用户需要修改

# 模型文件路径配置
MODEL_PATH = '/media/raspberry/64E5-D9E51/tarek.pkl'           # 主分类模型
SCALER_PATH = '/media/raspberry/64E5-D9E51/scaler81.pkl'       # 数据标准化器
LABEL_ENCODER_PATH = '/media/raspberry/64E5-D9E51/labelencoder81.pkl'  # 标签编码器
#Windows路径示例(取消注释使用)
#

# Windows路径示例(取消注释使用):
# MODEL_PATH = 'H:/models/tarek.pkl'
# SCALER_PATH = 'H:/models/scaler81.pkl'
# LABEL_ENCODER_PATH = 'H:/models/labelencoder81.pkl'

try:
    # 加载训练好的机器学习模型
    # 支持的模型类型: Random Forest, SVM, KNN, 神经网络等(sklearn兼容格式)
    model = load(MODEL_PATH)
    print(f"[INFO] 模型加载成功: {MODEL_PATH}")
    
    # 加载数据标准化器(用于特征缩放)
    # 注意: 必须使用训练时的相同scaler，否则预测结果会不准确
    scaler = load(SCALER_PATH)
    print(f"[INFO] 标准化器加载成功: {SCALER_PATH}")
    
    # 加载标签编码器(用于将数值标签转换为字母/单词)
    le = load(LABEL_ENCODER_PATH)
    print(f"[INFO] 标签编码器加载成功: {LABEL_ENCODER_PATH}")
    
    # 显示模型信息
    print(f"
[MODEL INFO] 模型类型: {type(model).__name__}")
    print(f"[MODEL INFO] 支持的类别: {list(le.classes_)}")
    
except FileNotFoundError as e:
    print(f"[ERROR] 模型文件未找到: {e}")
    print("[HINT] 请检查MODEL_PATH等路径配置是否正确")
    raise
except Exception as e:
    print(f"[ERROR] 加载模型时发生错误: {e}")
    raise

## 3. 配置参数设置

本单元格配置系统运行所需的各种参数。

### 网络配置:

- `HOST`: 服务器绑定的IP地址
  - `''` (空字符串): 监听所有网络接口
  - `'0.0.0.0'`: 同上，显式监听所有接口
  - `'192.168.x.x'`: 仅监听特定IP

- `PORT`: 服务器端口号 (默认4444)
  - 必须与ESP32固件中设置的端口号一致
  - 确保防火墙允许该端口通信

### TTS配置:

- `engine`: pyttsx3语音合成引擎实例
- 可配置参数: 语速(rate)、音量(volume)、语音(voice)

### 采样配置:

- `SAMPLE_DELAY`: 预测间隔(秒)，控制采样频率
- 默认1秒，可根据需要调整(0.5秒=2Hz, 0.1秒=10Hz)

In [ ]:
# ==========================================
# 第3部分: 配置参数设置
# ==========================================

# ---------- 网络配置 ----------
# HOST: 服务器监听地址
#   - '' 表示监听所有网络接口(推荐)
#   - 'localhost' 仅本地访问
#   - '192.168.x.x' 指定特定IP
HOST = ''

# PORT: 服务器端口号，必须与ESP32代码中的端口一致
PORT = 4444

# ---------- TTS(文字转语音)配置 ----------
# 初始化语音合成引擎
engine = pyttsx3.init()

# 可选: 配置TTS参数(取消注释使用)
# engine.setProperty('rate', 150)      # 语速 (默认200，范围50-300)
# engine.setProperty('volume', 0.9)    # 音量 (0.0-1.0)
# voices = engine.getProperty('voices')
# engine.setProperty('voice', voices[0].id)  # 选择语音(0=男声, 1=女声等)

# ---------- 采样配置 ----------
SAMPLE_DELAY = 1.0  # 预测间隔(秒)，可根据需要调整

# ---------- 数据包配置 ----------
# ESP32发送的数据包大小: 44字节
# 5个float(弯曲值) + 3个float(陀螺仪) + 3个float(加速度计) = 11*4 = 44字节
PACKET_SIZE = 44

# 特征列名定义(与训练时使用的特征顺序一致)
FEATURE_COLUMNS = [
    'flex_1', 'flex_2', 'flex_3', 'flex_4', 'flex_5',  # 5个弯曲传感器
    'GYRx', 'GYRy', 'GYRz',                              # 3轴陀螺仪
    'ACCx', 'ACCy', 'ACCz'                               # 3轴加速度计
]

print("[CONFIG] 系统参数配置完成:")
print(f"  - 监听地址: {HOST if HOST else '所有接口'}:{PORT}")
print(f"  - 数据包大小: {PACKET_SIZE}字节")
print(f"  - 采样间隔: {SAMPLE_DELAY}秒")
print(f"  - 特征数量: {len(FEATURE_COLUMNS)}维")
print(f"  - 特征列表: {FEATURE_COLUMNS}")

## 4. Socket服务器设置

本单元格创建TCP Socket服务器，用于接收ESP32发送的传感器数据。

### 通信流程:

```
1. 创建Socket (AF_INET=IPv4, SOCK_STREAM=TCP)
2. 绑定地址和端口 (bind)
3. 开始监听 (listen)
4. 等待ESP32连接 (accept)
5. 接收数据 (recv)
6. 解析数据 (parsing)
7. 预测 (predict)
8. 发送结果 (send)
9. 重复步骤4-8
10. 关闭连接 (close)
```

### 注意事项:

- 确保ESP32代码中的IP地址和端口号与服务器配置一致
- 确保防火墙允许该端口通信
- 服务器运行时，请确保ESP32已连接到网络并处于运行状态

In [ ]:
# ==========================================
# 第4部分: Socket服务器设置
# ==========================================

import socket
import time
import numpy as np

def predict(model, scaler, le, data):
    # 预测函数
    # data: 传感器数据
    # 返回: 预测结果
    
    # 数据预处理
    data = np.array(data).reshape(1, -1)  # 转换为1行n列的数组
    data = scaler.transform(data)  # 数据标准化
    
    # 模型预测
    prediction = model.predict(data)
    
    # 预测结果解码
    prediction = le.inverse_transform(prediction)
    
    return prediction

def start_server(model, scaler, le):
    # 服务器启动函数
    # model: 训练好的模型
    # scaler: 标准化器
    # le: 标签编码器
    
    # 创建Socket
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    
    # 绑定地址和端口
    server_socket.bind((HOST, PORT))
    
    # 开始监听
    server_socket.listen(1)
    print("[SERVER] 服务器已启动，等待ESP32连接...")
    
    while True:
        # 等待ESP32连接
        client_socket, client_address = server_socket.accept()
        print(f"[SERVER] ESP32已连接: {client_address}")
        
        while True:
            # 接收数据
            data = client_socket.recv(PACKET_SIZE)
            if not data:
                break
            
            # 解析数据
            data = data.decode().split(',')
            data = [float(d) for d in data]
            
            # 预测
            prediction = predict(model, scaler, le, data)
            
            # 发送结果
            client_socket.send(str(prediction[0]).encode())
        
        # 关闭连接
        client_socket.close()
    
    # 关闭服务器
    server_socket.close()


In [ ]:
# ==========================================
# 第4部分: Socket服务器设置
# ==========================================

import socket
import time
import numpy as np

def predict(model, scaler, le, data):
    # 预测函数
    # data: 传感器数据
    # 返回: 预测结果
    
    # 数据预处理
    data = np.array(data).reshape(1, -1)  # 转换为1行n列的数组
    data = scaler.transform(data)  # 数据标准化
    
    # 模型预测
    prediction = model.predict(data)
    
    # 预测结果解码
    prediction = le.inverse_transform(prediction)
    
    return prediction

def start_server(model, scaler, le):
    # 服务器启动函数
    # model: 训练好的模型
    # scaler: 标准化器
    # le: 标签编码器
    
    # 创建Socket
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    
    # 绑定地址和端口
    server_socket.bind((HOST, PORT))
    
    # 开始监听
    server_socket.listen(1)
    print("[SERVER] 服务器已启动，等待ESP32连接...")
    
    while True:
        # 等待ESP32连接
        client_socket, client_address = server_socket.accept()
        print(f"[SERVER] ESP32已连接: {client_address}")
        
        while True:
            # 接收数据
            data = client_socket.recv(PACKET_SIZE)
            if not data:
                break
            
            # 解析数据
            data = data.decode().split(',')
            data = [float(d) for d in data]
            
            # 预测
            prediction = predict(model, scaler, le, data)
            
            # 发送结果
            client_socket.send(str(prediction[0]).encode())
        
        # 关闭连接
        client_socket.close()
    
    # 关闭服务器
    server_socket.close()
